In [22]:
# === Cross-Decomposer Comparison ===
# Compare gulps vs nuop vs bqskit across ISAs.
# Metrics: end-to-end time, total cost (sum of gate strengths), fidelity.

import time

import numpy as np
import matplotlib.pyplot as plt
import scienceplots
import lovelyplots
from tqdm import tqdm
from qiskit import QuantumCircuit
from qiskit.quantum_info import average_gate_fidelity, random_unitary, Operator
from qiskit.circuit.library import iSwapGate

from gulps import GulpsDecomposer, GateInvariants, logger
from gulps.comparisons.nuop_driver import (
    xy_gate,
    ParallelGateReplacementPass,
)
from gulps.comparisons.bqskit_driver import Sqrt3iSwapGate, Sqrt4iSwapGate

from bqskit.ir.gates import SqrtISwapGate, U3Gate
from bqskit.qis.unitary.unitarymatrix import UnitaryMatrix
from bqskit import MachineModel, compile as bqskit_compile
from bqskit._logging import disable_logging

logger.setLevel("WARNING")
disable_logging()

In [23]:
N_SAMPLES = 50
SEED_OFFSET = 0
DECOMPOSERS = ["gulps", "nuop", "bqskit"]
LABELS = {"gulps": "GULPS", "nuop": "NuOp", "bqskit": "BQSKit"}
COLORS = {"gulps": "tab:blue", "nuop": "tab:orange", "bqskit": "tab:green"}

# --- ISA definitions ---
# Each ISA maps the same physical gates into each decomposer's native format.
# NuOp uses ParallelGateReplacementPass which natively handles hetero gate sets:
#   it tries every gate type for each 2Q operation and picks the best fidelity.
ISA_CONFIGS = [
    # ── Homogeneous ISAs ──
    {
        "name": r"iSwap$^{1/2}$",
        "gulps": {
            "gate_set": [iSwapGate().power(1 / 2)],
            "costs": [0.5],
            "names": ["sq2iswap"],
        },
        "nuop": {
            "gate_defs": [xy_gate],
            "gate_params": [[np.pi / 2]],
            "gate_labels": ["sq2iswap"],
            "costs": {"sq2iswap": 0.5},
        },
        "bqskit": {
            "gate_set": {SqrtISwapGate(), U3Gate()},
            "cost_map": {SqrtISwapGate: 0.5},
        },
    },
    {
        "name": r"iSwap$^{1/3}$",
        "gulps": {
            "gate_set": [iSwapGate().power(1 / 3)],
            "costs": [1 / 3],
            "names": ["sq3iswap"],
        },
        "nuop": {
            "gate_defs": [xy_gate],
            "gate_params": [[np.pi / 3]],
            "gate_labels": ["sq3iswap"],
            "costs": {"sq3iswap": 1 / 3},
        },
        "bqskit": {
            "gate_set": {Sqrt3iSwapGate(), U3Gate()},
            "cost_map": {Sqrt3iSwapGate: 1 / 3},
        },
    },
    {
        "name": r"iSwap$^{1/4}$",
        "gulps": {
            "gate_set": [iSwapGate().power(1 / 4)],
            "costs": [0.25],
            "names": ["sq4iswap"],
        },
        "nuop": {
            "gate_defs": [xy_gate],
            "gate_params": [[np.pi / 4]],
            "gate_labels": ["sq4iswap"],
            "costs": {"sq4iswap": 0.25},
        },
        "bqskit": {
            "gate_set": {Sqrt4iSwapGate(), U3Gate()},
            "cost_map": {Sqrt4iSwapGate: 0.25},
        },
    },
    # ── Heterogeneous ISAs ──
    {
        "name": r"iSwap$^{1/2}$ + iSwap$^{1/3}$",
        "gulps": {
            "gate_set": [iSwapGate().power(1 / 2), iSwapGate().power(1 / 3)],
            "costs": [0.5, 1 / 3],
            "names": ["sq2iswap", "sq3iswap"],
        },
        "nuop": {
            "gate_defs": [xy_gate, xy_gate],
            "gate_params": [[np.pi / 2], [np.pi / 3]],
            "gate_labels": ["sq2iswap", "sq3iswap"],
            "costs": {"sq2iswap": 0.5, "sq3iswap": 1 / 3},
        },
        "bqskit": {
            "gate_set": {SqrtISwapGate(), Sqrt3iSwapGate(), U3Gate()},
            "cost_map": {SqrtISwapGate: 0.5, Sqrt3iSwapGate: 1 / 3},
        },
    },
    {
        "name": r"iSwap$^{1/2}$ + iSwap$^{1/3}$ + iSwap$^{1/4}$",
        "gulps": {
            "gate_set": [
                iSwapGate().power(1 / 2),
                iSwapGate().power(1 / 3),
                iSwapGate().power(1 / 4),
            ],
            "costs": [0.5, 1 / 3, 0.25],
            "names": ["sq2iswap", "sq3iswap", "sq4iswap"],
        },
        "nuop": {
            "gate_defs": [xy_gate, xy_gate, xy_gate],
            "gate_params": [[np.pi / 2], [np.pi / 3], [np.pi / 4]],
            "gate_labels": ["sq2iswap", "sq3iswap", "sq4iswap"],
            "costs": {"sq2iswap": 0.5, "sq3iswap": 1 / 3, "sq4iswap": 0.25},
        },
        "bqskit": {
            "gate_set": {SqrtISwapGate(), Sqrt3iSwapGate(), Sqrt4iSwapGate(), U3Gate()},
            "cost_map": {
                SqrtISwapGate: 0.5,
                Sqrt3iSwapGate: 1 / 3,
                Sqrt4iSwapGate: 0.25,
            },
        },
    },
]

In [24]:
def make_nuop_decomposer(nuop_cfg):
    """Build a ParallelGateReplacementPass from ISA config."""
    n_gates = len(nuop_cfg["gate_defs"])
    return ParallelGateReplacementPass(
        gate_defs=nuop_cfg["gate_defs"],
        gate_params=nuop_cfg["gate_params"],
        gate_labels=nuop_cfg["gate_labels"],
        fidelity_dict_2q_gate={(0, 1): [1.0] * n_gates},
        fidelity_list_1q_gate=[1.0, 1.0],
        tol=0.0,
    )


def nuop_cost(output_qc, cost_map):
    """Sum gate strengths from a NuOp output circuit using gate labels."""
    total = 0.0
    for inst in output_qc:
        label = inst.operation.label
        if label is not None and label in cost_map:
            total += cost_map[label]
    return total


def bqskit_cost(bqskit_out, cost_map):
    """Sum gate strengths from a BQSKit output circuit."""
    total = 0.0
    for gate, count in bqskit_out.gate_counts.items():
        if gate.num_qudits == 2:
            total += count * cost_map.get(type(gate), 0)
    return total

In [25]:
from collections import Counter


def sentence_label(sentence):
    """Turn a GULPS sentence tuple into a readable string like '2×sq2iswap'."""
    counts = Counter(g.name for g in sentence)
    parts = [f"{v}×{k}" for k, v in counts.items()]
    return " + ".join(parts)


all_results = {}

for isa_cfg in ISA_CONFIGS:
    isa_name = isa_cfg["name"]
    print(f"\n=== ISA: {isa_name} ===")

    results = {d: {"times": [], "costs": [], "fidelities": []} for d in DECOMPOSERS}
    results["sentences"] = []

    seeds = range(SEED_OFFSET, SEED_OFFSET + N_SAMPLES)

    # ── GULPS (tight loop) ──
    g = isa_cfg["gulps"]
    gulps_dec = GulpsDecomposer(
        gate_set=g["gate_set"], costs=g["costs"], names=g["names"]
    )
    _ = gulps_dec(random_unitary(4, seed=999))  # warmup JIT

    for idx in tqdm(seeds, desc=f"{isa_name} / GULPS"):
        u = random_unitary(4, seed=idx)

        t0 = time.perf_counter()
        qc = gulps_dec(u)
        t1 = time.perf_counter()

        fid = average_gate_fidelity(u, Operator(qc))
        results["gulps"]["times"].append(t1 - t0)
        results["gulps"]["fidelities"].append(fid)

        # Extract optimal cost/sentence *after* timing to avoid warming caches
        target_inv = GateInvariants.from_unitary(u, enforce_alcove=True)
        optimal = gulps_dec._best_decomposition(target_inv)
        results["gulps"]["costs"].append(optimal.cost)
        results["sentences"].append(sentence_label(optimal.sentence))

    # ── NuOp (tight loop) ──
    nuop_dec = make_nuop_decomposer(isa_cfg["nuop"])
    _wqc = QuantumCircuit(2)
    _wqc.append(random_unitary(4, seed=999), [0, 1])
    _ = nuop_dec.run(_wqc, exact_decom=True)  # warmup

    for idx in tqdm(seeds, desc=f"{isa_name} / NuOp"):
        input_qc = QuantumCircuit(2)
        input_qc.append(random_unitary(4, seed=idx), [0, 1])

        t0 = time.perf_counter()
        nuop_qc = nuop_dec.run(input_qc, exact_decom=True)
        t1 = time.perf_counter()

        fid = average_gate_fidelity(Operator(input_qc), Operator(nuop_qc))
        cost = nuop_cost(nuop_qc, isa_cfg["nuop"]["costs"])
        results["nuop"]["times"].append(t1 - t0)
        results["nuop"]["costs"].append(cost)
        results["nuop"]["fidelities"].append(fid)

    # ── BQSKit (tight loop) ──
    bqskit_model = MachineModel(2, gate_set=isa_cfg["bqskit"]["gate_set"])
    _ = bqskit_compile(
        UnitaryMatrix(random_unitary(4, seed=999).data), model=bqskit_model
    )

    for idx in tqdm(seeds, desc=f"{isa_name} / BQSKit"):
        u_matrix = np.array(random_unitary(4, seed=idx).to_matrix())

        t0 = time.perf_counter()
        bqs_out = bqskit_compile(UnitaryMatrix(u_matrix), model=bqskit_model)
        t1 = time.perf_counter()

        cost = bqskit_cost(bqs_out, isa_cfg["bqskit"]["cost_map"])
        fid = average_gate_fidelity(
            Operator(u_matrix), Operator(bqs_out.get_unitary()._utry)
        )
        results["bqskit"]["times"].append(t1 - t0)
        results["bqskit"]["costs"].append(cost)
        results["bqskit"]["fidelities"].append(fid)

    all_results[isa_name] = results


=== ISA: iSwap$^{1/2}$ ===


iSwap$^{1/2}$ / BQSKit: 100%|██████████| 50/50 [02:04<00:00,  2.50s/it]



=== ISA: iSwap$^{1/3}$ ===


iSwap$^{1/3}$ / BQSKit: 100%|██████████| 50/50 [06:54<00:00,  8.29s/it]



=== ISA: iSwap$^{1/4}$ ===


iSwap$^{1/4}$ / BQSKit: 100%|██████████| 50/50 [07:45<00:00,  9.31s/it]



=== ISA: iSwap$^{1/2}$ + iSwap$^{1/3}$ ===


iSwap$^{1/2}$ + iSwap$^{1/3}$ / GULPS: 100%|██████████| 50/50 [00:00<00:00, 191.97it/s]
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
iSwap$^{1/2}$ + iSwap$^{1/3}$ / BQSKit: 100%|██████████| 50/50 [06:16<00:00,  7.52s/it]



=== ISA: iSwap$^{1/2}$ + iSwap$^{1/3}$ + iSwap$^{1/4}$ ===


iSwap$^{1/2}$ + iSwap$^{1/3}$ + iSwap$^{1/4}$ / GULPS: 100%|██████████| 50/50 [00:00<00:00, 65.48it/s]
iSwap$^{1/2}$ + iSwap$^{1/3}$ + iSwap$^{1/4}$ / NuOp: 100%|██████████| 50/50 [02:59<00:00,  3.60s/it]
iSwap$^{1/2}$ + iSwap$^{1/3}$ + iSwap$^{1/4}$ / BQSKit: 100%|██████████| 50/50 [07:11<00:00,  8.62s/it]


In [26]:
for isa_cfg in ISA_CONFIGS:
    isa_name = isa_cfg["name"]
    results = all_results[isa_name]
    print(f"\n=== {isa_name} ===")
    print(
        f"{'Decomposer':<12} {'Mean Time':>10} {'Med Time':>10}"
        f" {'Mean Cost':>10} {'Mean 1-F':>12} {'Max 1-F':>12}"
    )
    print("-" * 71)
    for d in DECOMPOSERS:
        r = results[d]
        if not r["times"]:
            print(f"{LABELS[d]:<12} {'(no data)':>10}")
            continue
        infids = 1 - np.array(r["fidelities"])
        print(
            f"{LABELS[d]:<12} "
            f"{np.mean(r['times']):>9.4f}s "
            f"{np.median(r['times']):>9.4f}s "
            f"{np.mean(r['costs']):>10.3f} "
            f"{np.mean(infids):>12.2e} "
            f"{np.max(infids):>12.2e}"
        )


=== iSwap$^{1/2}$ ===
Decomposer    Mean Time   Med Time  Mean Cost     Mean 1-F      Max 1-F
-----------------------------------------------------------------------
GULPS           0.0045s    0.0024s      1.160    -6.66e-18     1.11e-15
NuOp            0.6133s    0.6426s      1.380     1.31e-08     6.19e-07
BQSKit          2.4945s    2.5614s      1.430     3.22e-16     8.88e-16

=== iSwap$^{1/3}$ ===
Decomposer    Mean Time   Med Time  Mean Cost     Mean 1-F      Max 1-F
-----------------------------------------------------------------------
GULPS           0.0070s    0.0061s      0.987    -4.15e-16     1.11e-15
NuOp            1.0823s    0.9295s      1.107     1.15e-08     5.36e-07
BQSKit          8.2904s    8.2891s      1.193     1.04e-16     6.66e-16

=== iSwap$^{1/4}$ ===
Decomposer    Mean Time   Med Time  Mean Cost     Mean 1-F      Max 1-F
-----------------------------------------------------------------------
GULPS           0.0111s    0.0098s      0.945    -3.09e-16     1.55

In [1]:
n_isas = len(ISA_CONFIGS)
INFIDELITY_FLOOR = 1e-16

ISA_SHORT = [
    r"$\sqrt{}$",
    r"$\sqrt[3]{}$",
    r"$\sqrt[4]{}$",
    r"$\sqrt{}+\sqrt[3]{}$",
    r"$\sqrt{}+\sqrt[3]{}+\sqrt[4]{}$",
]

with plt.style.context(["science", "ieee", "use_mathtext"]):
    fig, (ax_time, ax_fid, ax_cost) = plt.subplots(1, 3, figsize=(7.16, 2.2), dpi=200)

    x = np.arange(n_isas)
    isa_names = [cfg["name"] for cfg in ISA_CONFIGS]
    w = 0.22

    # ── (a) Compile time — absolute, grouped bars ──
    for i, d in enumerate(DECOMPOSERS):
        vals = [np.median(all_results[name][d]["times"]) for name in isa_names]
        ax_time.bar(
            x + (i - 1) * w,
            vals,
            w,
            color=COLORS[d],
            edgecolor="white",
            linewidth=0.4,
            label=LABELS[d],
        )
    ax_time.set_yscale("log")
    ax_time.set_ylabel("Median time (s)")
    ax_time.set_title("(a) Compile time", fontsize=8)
    ax_time.set_xticks(x)
    ax_time.set_xticklabels(ISA_SHORT, fontsize=6)
    ax_time.tick_params(axis="y", labelsize=6)
    ax_time.set_xlabel("iSwap fractional power", fontsize=6)

    # ── (b) Infidelity — boxplots pooled across ISAs ──
    infid_data = []
    for d in DECOMPOSERS:
        pool = []
        for name in isa_names:
            pool.extend(
                np.clip(
                    1 - np.array(all_results[name][d]["fidelities"]),
                    INFIDELITY_FLOOR,
                    None,
                )
            )
        infid_data.append(pool)
    bp = ax_fid.boxplot(
        infid_data,
        tick_labels=[LABELS[d] for d in DECOMPOSERS],
        patch_artist=True,
        widths=0.5,
        flierprops=dict(marker="o", markersize=2, alpha=0.5),
        medianprops=dict(color="black", linewidth=0.8),
    )
    for patch, d in zip(bp["boxes"], DECOMPOSERS):
        patch.set_facecolor(COLORS[d])
        patch.set_alpha(0.7)
        patch.set_edgecolor("black")
        patch.set_linewidth(0.4)
    ax_fid.set_yscale("log")
    ax_fid.autoscale()
    ax_fid.set_ylabel("Infidelity ($1 - F$)")
    ax_fid.set_title("(b) Synthesis infidelity", fontsize=8)
    ax_fid.tick_params(axis="both", labelsize=6)

    # ── (c) Expected cost — grouped bars ──
    for i, d in enumerate(DECOMPOSERS):
        vals = [np.mean(all_results[name][d]["costs"]) for name in isa_names]
        ax_cost.bar(
            x + (i - 1) * w,
            vals,
            w,
            color=COLORS[d],
            edgecolor="white",
            linewidth=0.4,
            label=LABELS[d],
        )
    ax_cost.set_ylabel(r"$\mathbb{E}[\Sigma\;\mathrm{gate\;strength}]$")
    ax_cost.set_title("(c) Expected cost", fontsize=8)
    ax_cost.set_xticks(x)
    ax_cost.set_xticklabels(ISA_SHORT, fontsize=6)
    ax_cost.tick_params(axis="y", labelsize=6)
    ax_cost.set_xlabel("iSwap fractional power", fontsize=6)

    # Shared legend
    handles, labs = ax_time.get_legend_handles_labels()
    fig.legend(
        handles,
        labs,
        loc="upper center",
        ncol=3,
        fontsize=7,
        frameon=True,
        bbox_to_anchor=(0.5, 1.06),
    )

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    # plt.savefig("comparison_benchmarks.pdf", bbox_inches="tight")
    plt.show()

NameError: name 'ISA_CONFIGS' is not defined